# 04 - EM for Diffusion and Jumps

Goal: separate smooth diffusion from jump shocks in $\Delta x_t$.

Mixture model:
$$
\Delta x_t \sim (1-\lambda_t)\,\mathcal N(\mu_t,\sigma_{b,t}^2) + \lambda_t\,f_J(\cdot)
$$
with EM updates on responsibilities $\gamma_t = P(\text{jump}_t | \Delta x_t)$.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
inp = Path('stage3_kalman.csv')
if not inp.exists():
    raise FileNotFoundError('Run notebook 03 first to generate stage3_kalman.csv')

df = pd.read_csv(inp)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
df = df.sort_values('timestamp').reset_index(drop=True)
df['dx'] = df['x_hat'].diff().fillna(0.0)
df.head()

,timestamp,p_clipped,y_logit,sigma2_eps,x_hat,kalman_var,innovation,kalman_gain,dx
0,2025-11-14 16:31:16+00:00,0.977495,3.771256,0.0,3.771256,9.999500e-09,0.0,0.99995,0.0


## EM implementation (Gaussian diffusion + Gaussian jump proxy)

In [3]:
def gaussian_pdf(x, mu, var):
    var = np.maximum(var, 1e-10)
    z = (x - mu) ** 2 / var
    return np.exp(-0.5 * z) / np.sqrt(2 * np.pi * var)

dx = df['dx'].to_numpy()
n = len(dx)

# Init
mu = np.median(dx)
sigma2_b = np.var(dx) * 0.5 + 1e-6
sigma2_j = np.var(dx) * 4.0 + 1e-6
lam = 0.05

for _ in range(30):
    # E-step
    p_diff = (1 - lam) * gaussian_pdf(dx, mu, sigma2_b)
    p_jump = lam * gaussian_pdf(dx, 0.0, sigma2_j)
    gamma = p_jump / (p_jump + p_diff + 1e-18)

    # M-step
    w_diff = 1 - gamma
    w_jump = gamma

    mu = np.sum(w_diff * dx) / (np.sum(w_diff) + 1e-12)
    sigma2_b = np.sum(w_diff * (dx - mu) ** 2) / (np.sum(w_diff) + 1e-12)
    sigma2_j = np.sum(w_jump * (dx - 0.0) ** 2) / (np.sum(w_jump) + 1e-12)
    lam = np.mean(gamma)

df['jump_prob'] = gamma
df['sigma2_b'] = sigma2_b
df['sigma2_j'] = sigma2_j
df['lambda_jump'] = lam

print({'mu': float(mu), 'sigma2_b': float(sigma2_b), 'sigma2_j': float(sigma2_j), 'lambda': float(lam)})

{'mu': 0.0, 'sigma2_b': 0.0, 'sigma2_j': 0.0, 'lambda': 0.050000000000000024}


## Jump-dominant flags

A common cutoff is $\gamma_t > 0.7$.

In [4]:
df['is_jump_dominant'] = df['jump_prob'] > 0.7
df[['jump_prob', 'is_jump_dominant']].head(), df['is_jump_dominant'].mean()

(   jump_prob  is_jump_dominant
 0       0.05             False,
 np.float64(0.0))

In [5]:
out_cols = [
    'timestamp', 'p_clipped', 'y_logit', 'x_hat', 'dx',
    'jump_prob', 'is_jump_dominant', 'sigma2_b', 'sigma2_j', 'lambda_jump',
]
df[out_cols].to_csv('stage4_em.csv', index=False)
print('saved:', Path('stage4_em.csv').resolve())
print('rows:', len(df))

saved: C:\Users\p\Documents\GitHub\volatility-estimator\research\stage4_em.csv
rows: 1
